# Input Grad Steering — DiT concat · d256 · window 4

**Question.** Same probe-gradient question as the transformer/GRU notebooks, plus the DiT-only mechanisms this
thread was built to reach: **probe-guided sampling** (per-step kicks during generation) and **pause–optimize–
resume latent steering** (drive the noisy iterate to a verified probe readout mid-ODE, then let the remaining
denoise steps finish). The GRU/transformer notebooks showed the probe gradient finds off-manifold (adversarial)
content; the diffusion sampler is the first editor in the repo with a built-in "make this a valid observation"
operator. Does that rescue the edit?

**Data / model provenance.** Model `9_dset4_dit_w4_d256` = **DiT concat · d256 · window 4 (4 ctx frames)**,
checkpoint `runs/dit/9_dset4_dit_w4_d256/best_model.pt` (best mean-mode val MSE 0.02445; row copied from
`../DiT/DIT_RUNS.md`). Dataset `datasets/4_fixed_refl_inview` (edit frame `ef`=20, obs noise 0.2). N=64 edits,
K=15 rollout. §4 metrics from `scripts/editability_metrics.py`; probes from `pim.extractors.fit_readability_probes`.
The (residual point × τ) readability map that motivates the latent-steering point lives in
`../DiT/dit_world_state.ipynb`. Note the context asymmetry vs the transformer notebook: this model carries 4
frames, `W16` carries all 20.

## Definitions

| term / metric | definition | units | better |
|---|---|---|---|
| **activations state (probe space)** | the DiT's `"activations"` state view: final-block token features at the current position, computed at τ=1 from the fixed noise bank (d256) | — | — |
| **residual point ℓ** | trunk residual stream via `resid_sink`: 0 = token embedding, 1–3 = input to blocks 2–4, 4 = final-block output | — | — |
| **linear / MLP probe R²** | held-out R² of the standard readability probes (80/20 by-sequence), target = positions of both objects (4 dims) | — | ↑ |
| **Input Grad Steering (n, λ)** | history-write editor, identical to the other notebooks: Adam (300 steps, lr 0.02) on δ over the newest **n** buffer frames (n=4 = this model's whole carried window), minimizing `‖A·act(buffer+δ) + b − target‖² + λ‖δ‖²`, input clamped to [0,1]; frozen standard linear probe on the activations state | — | — |
| **Guided sample @steps 0..M−1 (g)** | per-step classifier guidance: fresh-noise 8-step Euler with a kick `x ← x − g·Δτ·∇ₓ‖A·act(buffer ⊕ x) + b − target‖²` after every update (full schedule) or only while τ_next ≥ 0.5 (early-τ) | — | — |
| **Latent Grad Steering @(Lℓ, τ_pause)** | pause–optimize–resume: run the fresh-noise Euler ODE to τ_pause, then Adam (300 steps, lr 0.05, diff space, no clamp) on δ over the **iterate** until a linear probe **fit and held-out-verified at that same (ℓ, τ_pause) on Euler-iterate states** reads the target, then resume the remaining Euler steps; the resumed frame starts the rollout | — | — |
| **Unguided sample @step 0 (control)** | fresh-noise Euler sample at step 0, no steering — separates "sampling per se" from steering | — | — |
| **Render write @1 (oracle)** | newest buffer frame ← clean render of the edited world (`gt_edited`); one-frame offset, clean where the model expects noisy | — | — |
| **Δ_true / cos / angle / probe residual** | as in the transformer notebook (`gt_edited − obs[ef−1]`; shuffled-pair chance) | — | — |
| **Edit Index / zone RMSEs / GT-traj RMSE / fidelity** | canonical §4 set (`../METRICS_AND_EDITORS.md` §4) | — | see registry |

**Reading caveat for sampled arms:** a sampled/steered generation faithfully reproduces an observation-noise
realisation, so its zone RMSEs vs the clean render carry a ≈ noise-floor offset the deterministic arms don't
have; the **Edit Index** is the calibrated number for those arms. All observation-space errors are scored
against the **clean** render; rollout step 0 decodes frame `ef`.

In [ ]:
# [1] Setup + standard readability probes on the activations state.
import os, sys
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models.loader import load_checkpoint, load_dataset
from pim.world_models.dit import DiTState
from pim.extractors import fit_readability_probes
from pim.figures.theme import style_ax
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ = 2
N_EDIT, K, N_PROBE = 64, 15, 800
STEER_STEPS, STEER_LR = 300, 0.02
LAM_MAIN = 0.1
OUT = "/tmp/input_grad_steering_dit"; os.makedirs(OUT, exist_ok=True)

MODEL_LABEL = "DiT concat · d256 · window 4"
model, info = load_checkpoint("../../../../runs/dit/9_dset4_dit_w4_d256/best_model.pt", device=DEVICE)
bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
ef, R = edits.edit_frame, edits.obs_res
sim = test.config["dataset"]["sim"]
W = model.cfg.window
print(f"model {MODEL_LABEL} | epoch {info.epoch} best mean-mode val MSE {info.val_loss:.5f} | "
      f"d_model {model.cfg.d_model} window {W} | ef={ef} R={R} | N_EDIT={N_EDIT} K={K} device={DEVICE}")

obs_probe = torch.from_numpy(test.obs[:N_PROBE]).float().to(DEVICE)
Tm1 = test.obs.shape[1] - 1
P_tgt = test.positions[:N_PROBE, :Tm1].reshape(N_PROBE, Tm1, N_OBJ * 2).astype(np.float32)
vis = test.is_visible[:N_PROBE, :Tm1, :N_OBJ].all(axis=2)
model.state_view = "activations"
acts = model.get_hidden_states(obs_probe).cpu().numpy()
PROBE = fit_readability_probes(acts, P_tgt, mask=vis, device=DEVICE)
display(Markdown(f"**Held-out position R² on the activations state** ({PROBE['n_train_seq']}/"
                 f"{PROBE['n_heldout_seq']} train/held-out sequences): linear **{PROBE['linear_r2']:.3f}**, "
                 f"MLP **{PROBE['mlp_r2']:.3f}**"))
A_t = torch.from_numpy(PROBE["A"]).to(DEVICE)
b_t = torch.from_numpy(PROBE["b"]).to(DEVICE)

In [ ]:
# [2] §4 machinery: edit zones, buffer states, references (unsteered + oracle render write).
N = min(N_EDIT, edits.n_samples)
oe = edits.edit_object[:N].astype(int)
with h5py.File(edits.h5_path, "r") as f:
    pre_vel = f["velocities"][:N, ef - 1, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32)
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)
target4 = torch.from_numpy(tgt_pos.reshape(N, N_OBJ * 2)).float().to(DEVICE)
gt_roll = edits.clean_obs[:N, ef:ef + K, :].astype(np.float32)

ZONES = build_edit_zones(pre_pos=pre_pos, tgt_pos=tgt_pos, pre_vel=pre_vel,
                         edit_object=oe, sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N, ef:ef + K, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
teleport = ZONES.teleport
gt_edited_t = torch.from_numpy(ZONES.gt_edited).float().to(DEVICE)

obs_e = torch.from_numpy(edits.obs[:N]).float().to(DEVICE)
base_frame = obs_e[:, ef - 1]
delta_true = ZONES.gt_edited - base_frame.cpu().numpy()
full_len = torch.full((N,), W, dtype=torch.long, device=DEVICE)

def state_from_frames(frames):
    """DiTState from the last W raw frames (ef=20 > W so the buffer is always full)."""
    return DiTState(frames[:, -W:].contiguous(), full_len)

def activations(state):
    """Differentiable activations-view flat state (the probe space)."""
    prev = model.state_view
    model.state_view = "activations"
    z = model.flat_state(state)
    model.state_view = prev
    return z

@torch.no_grad()
def rollout_from(state, k=K):
    """Mean-mode free-run: step 0 decodes frame ef."""
    x = model.decode(state); out = [x]; s = state
    for _ in range(k - 1):
        x, s = model.step(x, s)
        out.append(x)
    return torch.stack(out, 1).cpu().numpy()

hist = obs_e[:, :ef]                                   # frames 0..ef-1
state_unsteered = state_from_frames(hist)
CARDS, ROLLS = {}, {}
ROLLS["Unsteered"] = rollout_from(state_unsteered)
CARDS["Unsteered"] = edit_scorecard(ROLLS["Unsteered"], ZONES, gt_roll)

hist_oracle = torch.cat([hist[:, :-1], gt_edited_t.unsqueeze(1)], dim=1)
ROLLS["Render write @1 (oracle)"] = rollout_from(state_from_frames(hist_oracle))
CARDS["Render write @1 (oracle)"] = edit_scorecard(ROLLS["Render write @1 (oracle)"], ZONES, gt_roll)
print(f"N={N} edits | mean teleport {teleport.mean():.2f} sim-units | "
      f"unsteered Edit Index {CARDS['Unsteered']['edit_index']:+.2f} | "
      f"oracle render-write Edit Index {CARDS['Render write @1 (oracle)']['edit_index']:+.2f}")

In [ ]:
# [3] Editor A — Input Grad Steering (n, λ): identical mechanism to the transformer/GRU notebooks.
#     n = how many of the newest history frames δ may touch (n=4 = this model's whole carried window).
def input_grad_steer(lam, n=1, steps=STEER_STEPS, lr=STEER_LR):
    delta = torch.zeros(N, n, R, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    first_grad = None
    for it in range(steps):
        frames = torch.cat([hist[:, :ef - n], (hist[:, ef - n:] + delta).clamp(0, 1)], dim=1)
        z = activations(state_from_frames(frames))
        probe_loss = ((z @ A_t.T + b_t - target4) ** 2).sum(-1).mean()
        loss = probe_loss + lam * (delta ** 2).sum((-1, -2)).mean()
        opt.zero_grad(); loss.backward()
        if it == 0:
            first_grad = -delta.grad[:, -1].detach().clone()
        opt.step()
    with torch.no_grad():
        frames = torch.cat([hist[:, :ef - n], (hist[:, ef - n:] + delta).clamp(0, 1)], dim=1)
        state = state_from_frames(frames)
        resid_after = (activations(state) @ A_t.T + b_t - target4).norm(dim=-1).mean().item()
        resid_before = (activations(state_unsteered) @ A_t.T + b_t - target4).norm(dim=-1).mean().item()
    d_eff = (frames[:, -1] - base_frame).cpu().numpy()   # newest-frame delta (the Δ_true-comparable slice)
    return dict(delta=d_eff, first_grad=first_grad.cpu().numpy(),
                state=state, resid_before=resid_before, resid_after=resid_after)

ARMS = {}
arm_specs = [("Input Grad (n=1, λ=0.0)", dict(lam=0.0)),
             (f"Input Grad (n=1, λ={LAM_MAIN})", dict(lam=LAM_MAIN)),
             (f"Input Grad (n=4 whole window, λ={LAM_MAIN})", dict(lam=LAM_MAIN, n=W))]
for name, kw in arm_specs:
    ARMS[name] = input_grad_steer(**kw)
    ROLLS[name] = rollout_from(ARMS[name]["state"])
    CARDS[name] = edit_scorecard(ROLLS[name], ZONES, gt_roll)
    a = ARMS[name]
    print(f"{name:42s} probe residual {a['resid_before']:.3f} → {a['resid_after']:.3f} sim-units | "
          f"‖δ‖ (newest frame) {np.linalg.norm(a['delta'], axis=-1).mean():.3f} "
          f"(ref ‖Δ_true‖ {np.linalg.norm(delta_true, axis=-1).mean():.3f})")

def cos_rows(vecs):
    num = (vecs * delta_true).sum(-1)
    den = np.linalg.norm(vecs, axis=-1) * np.linalg.norm(delta_true, axis=-1) + 1e-12
    c = num / den
    return float(c.mean()), float(np.degrees(np.arccos(np.clip(c, -1, 1))).mean())

shuf = np.roll(ARMS[f"Input Grad (n=1, λ={LAM_MAIN})"]["delta"], 1, axis=0)
cos_chance, _ = cos_rows(shuf)
rows = []
for name, _ in arm_specs:
    a = ARMS[name]
    c_o, a_o = cos_rows(a["delta"]); c_g, a_g = cos_rows(a["first_grad"])
    rows.append(f"| {name} | {c_o:+.3f} | {a_o:.0f}° | {c_g:+.3f} | {a_g:.0f}° |")
display(Markdown("**Alignment with the true edit direction** (newest-frame δ; shuffled-pair chance cosine = "
                 f"{cos_chance:+.3f})\n\n| arm | cos(δ*, Δ_true) | angle | cos(first grad, Δ_true) | angle |\n"
                 "|---|---|---|---|---|\n" + "\n".join(rows)))

In [ ]:
# [4] Editor B — probe-guided sampling (classifier guidance in the Euler loop) + sampled control.
#     Two guidance schedules: FULL (kick after every Euler step, incl. the last — the naive version) and
#     EARLY-τ (kick only while τ_next ≥ 0.5, so the low-noise half of the ODE denoises freely — the actual
#     test of "the denoiser projects the perturbation back onto the manifold").
N_EULER = model.cfg.n_sample_steps   # 8

def guided_sample_frame(buffer, g, gen, guide=True, tau_stop=0.0):
    """One next-frame generation: fresh-noise Euler ODE with a probe-gradient kick after each update
    while τ_next ≥ tau_stop.  buffer (N, W, R) raw; returns (N, R) raw-space frame."""
    cur, nxt = model._window_tokens(buffer)                       # diff space, clean pairs
    attn = model._window_attn_mask(full_len, DEVICE)
    x = torch.randn(N, R, generator=gen).to(DEVICE)               # fresh start noise (today's rollout fix)
    taus = torch.linspace(1.0, 0.0, N_EULER + 1, device=DEVICE)
    for k in range(N_EULER):
        with torch.no_grad():
            nxt_k = torch.cat([nxt[:, :-1], x.unsqueeze(1)], dim=1)
            tau_k = torch.zeros(N, W, device=DEVICE); tau_k[:, -1] = taus[k]
            v = model._denoise(cur, nxt_k, tau_k, attn)[:, -1]
        dt = (taus[k + 1] - taus[k])
        x = x + dt * v
        if guide and taus[k + 1] >= tau_stop:
            xg = x.detach().requires_grad_(True)
            obs_x = model._from_diff(xg).clamp(0, 1)
            new_buf = torch.cat([buffer[:, 1:], obs_x.unsqueeze(1)], dim=1)
            z = activations(DiTState(new_buf, full_len))
            L = ((z @ A_t.T + b_t - target4) ** 2).sum()          # batch rows independent → per-sample grads
            gL = torch.autograd.grad(L, xg)[0]
            x = (xg - g * (-dt) * gL).detach()                    # -dt = Δτ > 0
    return model._from_diff(x).clamp(0, 1)

@torch.no_grad()
def _advance(state, x):
    return model.step(x, state)[1]

def rollout_guided(g, M, seed=123, guide=True, tau_stop=0.0, k=K):
    """First M frames: (guided) fresh-noise samples fed back; then mean-mode free-run."""
    gen = torch.Generator().manual_seed(seed)
    s = state_unsteered
    out = []
    for step in range(k):
        if step < M:
            x = guided_sample_frame(s.obs_buffer, g, gen, guide=guide, tau_stop=tau_stop)
        else:
            with torch.no_grad():
                x = model.decode(s)
        out.append(x.detach())
        s = _advance(s, x.detach())
    return torch.stack(out, 1).cpu().numpy()

FULL_G, EARLY_G = [1.0, 10.0, 100.0], [10.0, 100.0, 1000.0]
for g in FULL_G:
    name = f"Guided @step0, full sched. (g={g:g})"
    ROLLS[name] = rollout_guided(g, M=1)
    CARDS[name] = edit_scorecard(ROLLS[name], ZONES, gt_roll)
    print(f"{name:40s} Edit Index {CARDS[name]['edit_index']:+.2f} | "
          f"edit-frame RMSE {CARDS[name]['edit_frame_rmse']:.3f} | collateral {CARDS[name]['collateral_rmse']:.3f}")
for g in EARLY_G:
    name = f"Guided @step0, early-τ only (g={g:g})"
    ROLLS[name] = rollout_guided(g, M=1, tau_stop=0.5)
    CARDS[name] = edit_scorecard(ROLLS[name], ZONES, gt_roll)
    print(f"{name:40s} Edit Index {CARDS[name]['edit_index']:+.2f} | "
          f"edit-frame RMSE {CARDS[name]['edit_frame_rmse']:.3f} | collateral {CARDS[name]['collateral_rmse']:.3f}")

ROLLS["Unguided sample @step0 (control)"] = rollout_guided(0.0, M=1, guide=False)
CARDS["Unguided sample @step0 (control)"] = edit_scorecard(ROLLS["Unguided sample @step0 (control)"], ZONES, gt_roll)

# Multi-frame arm uses the early-τ schedule (the defensible one); g chosen by eye from the sweep above —
# NOT by max Edit Index, which rewards degradation (see CLAUDE.md on gaming the index).
G_MAIN = 100.0
name5 = f"Guided @steps0–4, early-τ only (g={G_MAIN:g})"
ROLLS[name5] = rollout_guided(G_MAIN, M=5, tau_stop=0.5)
CARDS[name5] = edit_scorecard(ROLLS[name5], ZONES, gt_roll)
print(f"{name5}: Edit Index {CARDS[name5]['edit_index']:+.2f} | "
      f"control (unguided sample): {CARDS['Unguided sample @step0 (control)']['edit_index']:+.2f}")

In [ ]:
# [5] Fig 2 — the two editors' outputs in observation space, per sample.
#     Col 1: Input Grad steered history frame. Col 2: early-τ guided vs unguided sample at step 0.
#     Col 3: full-schedule guided sample (the naive version) vs unguided — shows what late-τ kicks do.
SAMPLES = list(np.argsort(teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])
x_axis = np.arange(R)
ARM_EARLY = f"Guided @step0, early-τ only (g={G_MAIN:g})"
ARM_FULL = "Guided @step0, full sched. (g=100)"
guided_early_f0 = ROLLS[ARM_EARLY][:, 0]
guided_full_f0 = ROLLS[ARM_FULL][:, 0]
unguided_f0 = ROLLS["Unguided sample @step0 (control)"][:, 0]

fig, axes = plt.subplots(len(SAMPLES), 3, figsize=(4.6 * 3, 2.9 * len(SAMPLES)), sharex=True, sharey=True)
for r, smp in enumerate(SAMPLES):
    steered = base_frame[smp].cpu().numpy() + ARMS[f"Input Grad (n=1, λ={LAM_MAIN})"]["delta"][smp]
    panels = [
        (f"Input Grad steered obs[ef−1] (λ={LAM_MAIN})", base_frame[smp].cpu().numpy(), steered),
        (f"Early-τ guided (g={G_MAIN:g}) vs unguided, step 0", unguided_f0[smp], guided_early_f0[smp]),
        ("Full-sched. guided (g=100) vs unguided, step 0", unguided_f0[smp], guided_full_f0[smp]),
    ]
    for c, (title, base, out) in enumerate(panels):
        ax = axes[r, c]
        ax.plot(x_axis, base, color="#999999", lw=1.0)
        ax.plot(x_axis, ZONES.gt_edited[smp], color="#009E73", lw=1.4, ls="--")
        ax.plot(x_axis, out, color="#D55E00", lw=1.4)
        ax.fill_between(x_axis, 0, 1, where=ZONES.target[smp], color="#009E73", alpha=0.08)
        ax.fill_between(x_axis, 0, 1, where=ZONES.ghost[smp], color="#FF5252", alpha=0.08)
        ax.set_ylim(-0.05, 1.1)
        if r == 0: ax.set_title(title, fontsize=10)
        if c == 0: ax.set_ylabel(f"sample {smp}\n(teleport {teleport[smp]:.1f})\nintensity", fontsize=9)
        if r == len(SAMPLES) - 1: ax.set_xlabel("ray")
        style_ax(ax)
handles = [Line2D([0], [0], color="#999999", lw=1.5), Line2D([0], [0], color="#D55E00", lw=1.5),
           Line2D([0], [0], color="#009E73", ls="--", lw=1.5),
           Line2D([0], [0], color="#009E73", lw=6, alpha=0.2), Line2D([0], [0], color="#FF5252", lw=6, alpha=0.2)]
labels = ["base series (gray; named in column title)", "editor output (orange; named in column title)",
          "clean edited-world render", "target rays (shaded)", "ghost rays (shaded)"]
fig.legend(handles, labels, loc="upper center", ncol=5, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle(f"Fig 2 — editor outputs in observation space ({MODEL_LABEL}); column 1 edits history, "
             "columns 2–3 show the generated edit frame under the two guidance schedules", y=1.06, fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{OUT}/fig2_editor_outputs.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# [6] §4 scorecard for every arm + Fig 3 (Edit Index by rollout step).
ORDER = (["Unsteered", "Input Grad (n=1, λ=0.0)", f"Input Grad (n=1, λ={LAM_MAIN})",
          "Unguided sample @step0 (control)"]
         + [f"Guided @step0, full sched. (g={g:g})" for g in FULL_G]
         + [f"Guided @step0, early-τ only (g={g:g})" for g in EARLY_G]
         + [f"Guided @steps0–4, early-τ only (g={G_MAIN:g})", "Render write @1 (oracle)"])
hdr = ("| arm | Edit Index (−1…+1) | edit-frame RMSE | target RMSE | ghost RMSE | collateral RMSE "
       "| GT-traj RMSE | fidelity ratio |\n|---|---|---|---|---|---|---|---|\n")
rows = []
for name in ORDER:
    c = CARDS[name]
    rows.append(f"| {name} | {c['edit_index']:+.2f} | {c['edit_frame_rmse']:.3f} | {c['target_rmse']:.3f} "
                f"| {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} "
                f"| {fidelity_ratio(c, CARDS['Unsteered']):.2f} |")
display(Markdown("**§4 scorecard** (sampled arms carry an observation-noise floor in their RMSEs — the Edit "
                 "Index is the calibrated number for them; see Definitions. Watch the collateral column: an "
                 "index that moved while collateral exploded is degradation, not editing)\n\n" + hdr + "\n".join(rows)))

FIG3 = ["Unsteered", f"Input Grad (n=1, λ={LAM_MAIN})", "Unguided sample @step0 (control)",
        f"Guided @step0, early-τ only (g={G_MAIN:g})", f"Guided @steps0–4, early-τ only (g={G_MAIN:g})",
        "Render write @1 (oracle)"]
colors = ["#999999", "#56B4E9", "#CC79A7", "#0072B2", "#D55E00", "#009E73"]
fig, ax = plt.subplots(figsize=(8.5, 4))
for name, col in zip(FIG3, colors):
    ax.plot(CARDS[name]["edit_index_by_step"], color=col, lw=1.8, marker="o", ms=3.5, label=name)
ax.axhline(0, color="#555555", lw=0.8, ls=":")
ax.set_xlabel("rollout step (0 = edit frame ef)"); ax.set_ylabel("Edit Index (−1…+1)")
ax.set_ylim(-1.05, 1.05)
ax.set_title(f"Fig 3 — Edit Index by rollout step ({MODEL_LABEL})")
ax.legend(fontsize=8, loc="upper right"); style_ax(ax)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_index_by_step.png", dpi=150); plt.show()

In [ ]:
# [7] Fig 4 — observation-space waterfall (canonical fixed spec; one helper).
N_CTX = 6
ctx_obs = edits.obs[:N, ef - N_CTX:ef, :].astype(np.float32)
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TARGET_C, GHOST_C = "#00E676", "#FF5252"
def _cx(m):
    i = np.where(m)[0]; return i.mean() if i.size else np.nan
tgt_cx = np.array([_cx(ZONES.target[i]) for i in range(N)])
pre_cx = np.array([_cx(ZONES.ghost[i]) for i in range(N)])

def waterfall_grid(col_titles, col_bodies, samples, suptitle, fname):
    """col_bodies[c]: (N, L, R) rows BELOW the N_CTX context frames; every column its OWN free-run
    from step 0 (= frame ef). No shared teacher-forced ef row (banned; see CLAUDE.md)."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(samples), ncol, figsize=(3.0 * ncol, 3.4 * len(samples)),
                             squeeze=False, facecolor=DARK)
    for r, smp in enumerate(samples):
        for c in range(ncol):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx_obs[smp], col_bodies[c][smp]], axis=0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1,
                      interpolation="nearest")
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            ax.axhline(N_CTX - 0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(tgt_cx[smp]): ax.axvline(tgt_cx[smp], color=TARGET_C, lw=1.6, alpha=0.9)
            if not np.isnan(pre_cx[smp]): ax.axvline(pre_cx[smp], color=GHOST_C, ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(col_titles[c], fontsize=8, color=TXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (teleport {teleport[smp]:.1f})\nsim frame", fontsize=8, color=TXT)
                ax.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
                ax.set_yticklabels([ef - N_CTX, ef, ef + 7, ef + 14], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=TXT); ax.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0], [0], color=TARGET_C, lw=2.2, label="object target location"),
               Line2D([0], [0], color=GHOST_C, ls="--", lw=2.2, label="ghost (pre-edit) location"),
               Line2D([0], [0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here ({N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = frame {ef})")]
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.965))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(fname, dpi=150, facecolor=DARK, bbox_inches="tight"); plt.show()

WF = ["Unsteered", f"Input Grad (n=1, λ={LAM_MAIN})", "Unguided sample @step0 (control)",
      f"Guided @step0, early-τ only (g={G_MAIN:g})", f"Guided @steps0–4, early-τ only (g={G_MAIN:g})",
      "Guided @step0, full sched. (g=100)", "Render write @1 (oracle)"]
titles = ["GT (sim clean obs)"] + [f"{n}\nEdit Index {CARDS[n]['edit_index']:+.2f}" for n in WF]
bodies = [gt_roll] + [ROLLS[n] for n in WF]
waterfall_grid(titles, bodies, SAMPLES,
               f"Fig 4 — free-run waterfalls: input-grad and probe-guided-sampling arms vs references ({MODEL_LABEL})",
               f"{OUT}/fig4_waterfall.png")

In [ ]:
# [8] Editor C — Latent Grad Steering @(L=3, τ_pause): pause the ODE, optimize the ITERATE until a probe
#     fit AND VERIFIED at that same (residual point, τ) reads the target, then resume the remaining
#     denoise steps. The strict form of the original proposal. Steering point L=3 chosen from the
#     (residual point × τ) grid in ../../DiT/dit_world_state.ipynb (late block is the max, R² ≈ 0.70, flat in τ).
S = model.cfg.n_sample_steps
taus_full = torch.linspace(1.0, 0.0, S + 1, device=DEVICE)
L_STEER = 3
PAUSE_KS = [2, 4, 6]                                    # τ_pause = 0.75, 0.5, 0.25
N_PROBE_LAT = 500

# ── fit + verify the steering probes at (L=3, τ_pause) on test-split Euler-iterate states ──
obs_p = torch.from_numpy(test.obs[:N_PROBE_LAT]).float().to(DEVICE)
Bp, Tp, _ = obs_p.shape
win_p, len_p = model._unfold_windows(obs_p)
flat_wp = win_p.reshape(Bp * (Tp - 1), W, R)
flat_lp = len_p.unsqueeze(0).expand(Bp, -1).reshape(-1)
n_rows = flat_wp.shape[0]
lat_states = {k: np.empty((n_rows, model.cfg.d_model), np.float32) for k in PAUSE_KS}
genp = torch.Generator().manual_seed(7)
eps_rows = torch.randn(n_rows, R, generator=genp)
with torch.no_grad():
    for i0 in range(0, n_rows, 4096):
        sl = slice(i0, min(i0 + 4096, n_rows))
        winc, lnc = flat_wp[sl], flat_lp[sl]
        n = winc.shape[0]
        cur, nxt = model._window_tokens(winc)
        attn = model._window_attn_mask(lnc, DEVICE)
        x = eps_rows[sl].to(DEVICE)
        for k in range(S):
            tau_t = torch.zeros(n, W, device=DEVICE); tau_t[:, -1] = taus_full[k]
            nxt_k = torch.cat([nxt[:, :-1], x.unsqueeze(1)], dim=1)
            if k in PAUSE_KS:
                sink = []
                feats, c = model._trunk(cur, nxt_k, tau_t, attn, resid_sink=sink)
                lat_states[k][sl] = sink[L_STEER][:, -1].cpu().numpy()
                v = model.final_layer(feats, c)[:, -1]
            else:
                v = model._denoise(cur, nxt_k, tau_t, attn)[:, -1]
            x = x + (taus_full[k + 1] - taus_full[k]) * v

P_tgt_p = test.positions[:N_PROBE_LAT, :Tp - 1].reshape(N_PROBE_LAT, Tp - 1, N_OBJ * 2).astype(np.float32)
vis_p = test.is_visible[:N_PROBE_LAT, :Tp - 1, :N_OBJ].all(axis=2)
LAT_PROBES = {}
for k in PAUSE_KS:
    fit = fit_readability_probes(lat_states[k].reshape(N_PROBE_LAT, Tp - 1, -1), P_tgt_p, mask=vis_p,
                                 device=DEVICE)
    LAT_PROBES[k] = fit
    print(f"steering probe @ (L={L_STEER}, τ={taus_full[k]:.2f}): held-out linear R² = {fit['linear_r2']:.3f} "
          f"(MLP {fit['mlp_r2']:.3f}) — verified recovery at the steering point")

# ── the editor ──
def latent_grad_steer(k_pause, lam=0.0, steps=STEER_STEPS, lr=0.05, seed=456):
    """Euler to τ_pause on the edit buffers → Adam on δ over the ITERATE until the (L=3, τ_pause) probe
    reads the target → resume the remaining Euler steps → the resulting frame starts the rollout."""
    fit = LAT_PROBES[k_pause]
    A = torch.from_numpy(fit["A"]).to(DEVICE); b = torch.from_numpy(fit["b"]).to(DEVICE)
    buffer = state_unsteered.obs_buffer
    cur, nxt = model._window_tokens(buffer)
    attn = model._window_attn_mask(full_len, DEVICE)
    gen = torch.Generator().manual_seed(seed)
    x = torch.randn(N, R, generator=gen).to(DEVICE)

    def v_at(xc, k):
        tau_t = torch.zeros(N, W, device=DEVICE); tau_t[:, -1] = taus_full[k]
        nxt_k = torch.cat([nxt[:, :-1], xc.unsqueeze(1)], dim=1)
        return model._denoise(cur, nxt_k, tau_t, attn)[:, -1]

    def probe_read(xc):
        tau_t = torch.zeros(N, W, device=DEVICE); tau_t[:, -1] = taus_full[k_pause]
        nxt_k = torch.cat([nxt[:, :-1], xc.unsqueeze(1)], dim=1)
        sink = []
        model._trunk(cur, nxt_k, tau_t, attn, resid_sink=sink)
        return sink[L_STEER][:, -1] @ A.T + b

    with torch.no_grad():
        for k in range(k_pause):
            x = x + (taus_full[k + 1] - taus_full[k]) * v_at(x, k)
        resid_before = (probe_read(x) - target4).norm(dim=-1).mean().item()
    delta = torch.zeros_like(x, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    for it in range(steps):
        loss = ((probe_read(x + delta) - target4) ** 2).sum(-1).mean() + lam * (delta ** 2).sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        x = (x + delta).detach()
        resid_after = (probe_read(x) - target4).norm(dim=-1).mean().item()
        for k in range(k_pause, S):
            x = x + (taus_full[k + 1] - taus_full[k]) * v_at(x, k)
        frame = model._from_diff(x).clamp(0, 1)
    return frame, resid_before, resid_after, float(delta.norm(dim=-1).mean())

@torch.no_grad()
def rollout_from_frame(frame, k=K):
    """frame = the editor's step-0 output; feed it back, then mean-mode free-run."""
    out = [frame]; s = state_unsteered
    x = frame
    for _ in range(k - 1):
        x, s = model.step(x, s)
        out.append(x)
    return torch.stack(out, 1).cpu().numpy()

LAT_FRAMES = {}
for k in PAUSE_KS:
    name = f"Latent Grad @(L3, τ={taus_full[k]:.2f})"
    frame, r0, r1, dn = latent_grad_steer(k)
    LAT_FRAMES[name] = frame.cpu().numpy()
    ROLLS[name] = rollout_from_frame(frame)
    CARDS[name] = edit_scorecard(ROLLS[name], ZONES, gt_roll)
    print(f"{name:28s} probe residual {r0:.3f} → {r1:.3f} sim-units | ‖δ‖ (diff space) {dn:.3f} | "
          f"Edit Index {CARDS[name]['edit_index']:+.2f} | collateral {CARDS[name]['collateral_rmse']:.3f}")

In [ ]:
# [9] Fig 5 — latent-steered resumed frames vs references; scorecard addendum; Fig 6 waterfall.
unguided_f0 = ROLLS["Unguided sample @step0 (control)"][:, 0]
LAT_NAMES = [f"Latent Grad @(L3, τ={taus_full[k]:.2f})" for k in PAUSE_KS]

fig, axes = plt.subplots(len(SAMPLES), 3, figsize=(4.6 * 3, 2.9 * len(SAMPLES)), sharex=True, sharey=True)
for r, smp in enumerate(SAMPLES):
    for c, name in enumerate(LAT_NAMES):
        ax = axes[r, c]
        ax.plot(x_axis, unguided_f0[smp], color="#999999", lw=1.0)
        ax.plot(x_axis, ZONES.gt_edited[smp], color="#009E73", lw=1.4, ls="--")
        ax.plot(x_axis, LAT_FRAMES[name][smp], color="#D55E00", lw=1.4)
        ax.fill_between(x_axis, 0, 1, where=ZONES.target[smp], color="#009E73", alpha=0.08)
        ax.fill_between(x_axis, 0, 1, where=ZONES.ghost[smp], color="#FF5252", alpha=0.08)
        ax.set_ylim(-0.05, 1.1)
        if r == 0: ax.set_title(name, fontsize=10)
        if c == 0: ax.set_ylabel(f"sample {smp}\n(teleport {teleport[smp]:.1f})\nintensity", fontsize=9)
        if r == len(SAMPLES) - 1: ax.set_xlabel("ray")
        style_ax(ax)
handles = [Line2D([0], [0], color="#999999", lw=1.5), Line2D([0], [0], color="#D55E00", lw=1.5),
           Line2D([0], [0], color="#009E73", ls="--", lw=1.5),
           Line2D([0], [0], color="#009E73", lw=6, alpha=0.2), Line2D([0], [0], color="#FF5252", lw=6, alpha=0.2)]
labels = ["unguided sample (control, gray)", "latent-steered resumed frame (orange)",
          "clean edited-world render", "target rays (shaded)", "ghost rays (shaded)"]
fig.legend(handles, labels, loc="upper center", ncol=5, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle(f"Fig 5 — the frame produced by pause–optimize–resume latent steering, by pause time "
             f"({MODEL_LABEL}, probe at residual point 3)", y=1.06, fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{OUT}/fig5_latent_frames.png", dpi=150, bbox_inches="tight"); plt.show()

# scorecard addendum: the new arms next to the references
ORDER2 = (["Unsteered", f"Input Grad (n=1, λ={LAM_MAIN})", f"Input Grad (n=4 whole window, λ={LAM_MAIN})",
           "Unguided sample @step0 (control)"] + LAT_NAMES + ["Render write @1 (oracle)"])
hdr = ("| arm | Edit Index (−1…+1) | edit-frame RMSE | target RMSE | ghost RMSE | collateral RMSE "
       "| GT-traj RMSE | fidelity ratio |\n|---|---|---|---|---|---|---|---|\n")
rows = []
for name in ORDER2:
    c = CARDS[name]
    rows.append(f"| {name} | {c['edit_index']:+.2f} | {c['edit_frame_rmse']:.3f} | {c['target_rmse']:.3f} "
                f"| {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} "
                f"| {fidelity_ratio(c, CARDS['Unsteered']):.2f} |")
display(Markdown("**§4 scorecard — latent-steering and whole-window arms** (same conventions/caveats as the "
                 "cell [6] table)\n\n" + hdr + "\n".join(rows)))

WF2 = ["Unsteered", f"Input Grad (n=4 whole window, λ={LAM_MAIN})", "Unguided sample @step0 (control)"] \
      + LAT_NAMES + ["Render write @1 (oracle)"]
titles = ["GT (sim clean obs)"] + [f"{n}\nEdit Index {CARDS[n]['edit_index']:+.2f}" for n in WF2]
bodies = [gt_roll] + [ROLLS[n] for n in WF2]
waterfall_grid(titles, bodies, SAMPLES,
               f"Fig 6 — free-run waterfalls: pause–optimize–resume latent steering and whole-window input "
               f"steering vs references ({MODEL_LABEL})",
               f"{OUT}/fig6_latent_waterfall.png")

In [ ]:
# [10] Robustness — the same arms under a FULL Euler-sampled rollout: fresh-noise 8-step generation at EVERY
#      rollout step, no mean-mode continuation. Answers whether the mixed-mode readout (sampled edit frame +
#      mean-mode continuation) changed any verdict. Sampled rollouts carry the ≈0.2 noise floor in RMSEs;
#      the Edit Index is the calibrated number (Definitions).
def _restore_bank():
    g0 = torch.Generator().manual_seed(model.cfg.noise_seed)
    model._eps_bank.copy_(torch.randn(model.cfg.n_mean_eps, model.cfg.input_dim, generator=g0).to(DEVICE))

@torch.no_grad()
def full_euler_rollout(state, seed, first_frame=None, k=K):
    """Free-run where every frame is a fresh-noise Euler sample (predict_mode='sample').
    first_frame: optional externally produced step-0 frame (e.g. the latent-steered resumed frame)."""
    gen = torch.Generator().manual_seed(seed)
    model.predict_mode = "sample"
    s = state
    if first_frame is None:
        model._eps_bank[0].copy_(torch.randn(R, generator=gen).to(DEVICE))
        x = model.decode(s)
    else:
        x = first_frame
    out = [x]
    for _ in range(k - 1):
        model._eps_bank[0].copy_(torch.randn(R, generator=gen).to(DEVICE))
        x, s = model.step(x, s)
        out.append(x)
    model.predict_mode = "mean"; _restore_bank()
    return torch.stack(out, 1).cpu().numpy()

FE = {}
FE["Unsteered (full Euler)"] = full_euler_rollout(state_unsteered, seed=901)
FE[f"Input Grad (n=1, λ={LAM_MAIN}) (full Euler)"] = full_euler_rollout(
    ARMS[f"Input Grad (n=1, λ={LAM_MAIN})"]["state"], seed=902)
for k_p in PAUSE_KS:
    src = f"Latent Grad @(L3, τ={taus_full[k_p]:.2f})"
    frame = torch.from_numpy(LAT_FRAMES[src][:, ]).to(DEVICE) if isinstance(LAT_FRAMES[src], np.ndarray) \
        else LAT_FRAMES[src]
    FE[f"{src} (full Euler)"] = full_euler_rollout(state_unsteered, seed=903 + k_p, first_frame=frame)
FE["Render write @1 (oracle) (full Euler)"] = full_euler_rollout(
    model.state_from_obs(hist_oracle) if hasattr(model, "state_from_obs") else state_from_frames(hist_oracle),
    seed=910)

hdr = ("| arm | Edit Index (−1…+1) | edit-frame RMSE | collateral RMSE | GT-traj RMSE |\n"
       "|---|---|---|---|---|\n")
rows = []
for name, roll in FE.items():
    CARDS[name] = edit_scorecard(roll, ZONES, gt_roll)
    ROLLS[name] = roll
    c = CARDS[name]
    rows.append(f"| {name} | {c['edit_index']:+.2f} | {c['edit_frame_rmse']:.3f} "
                f"| {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} |")
display(Markdown("**§4 scorecard under full Euler-sampled rollouts** (compare each row against its mixed-mode "
                 "counterpart in the cell [6]/[9] tables — the mechanism verdicts should be mode-invariant)\n\n"
                 + hdr + "\n".join(rows)))

WF3 = ["Unsteered (full Euler)", f"Input Grad (n=1, λ={LAM_MAIN}) (full Euler)"] \
      + [f"Latent Grad @(L3, τ={taus_full[k_p]:.2f}) (full Euler)" for k_p in PAUSE_KS] \
      + ["Render write @1 (oracle) (full Euler)"]
titles = ["GT (sim clean obs)"] + [f"{n}\nEdit Index {CARDS[n]['edit_index']:+.2f}" for n in WF3]
bodies = [gt_roll] + [ROLLS[n] for n in WF3]
waterfall_grid(titles, bodies, SAMPLES,
               f"Fig 7 — free-run waterfalls under FULL Euler-sampled rollouts (fresh noise every step; "
               f"{MODEL_LABEL})",
               f"{OUT}/fig7_full_euler_waterfall.png")

## MLP-probe steering variants (definitions for cells [11]–[12])

Same editors, same targets, same optimization — the only change is the frozen probe: the **standard 2×256 ReLU
readability MLP** fit (and held-out-verified) at the same point as its linear counterpart — `PROBE["mlp"]` on
the activations state for Input Grad, `LAT_PROBES[k]["mlp"]` at (L3, τ_pause) on Euler-iterate states for
Latent Grad. After the 2026-08-11 `STD_EPOCHS` fix these probes read **more** than the linear ones (≈0.85 vs
≈0.70 held-out R² at the steering points), so this section asks: does a richer readout give the gradient a
better-shaped direction, or just a richer adversarial surface?

⚠ Not to be confused with the registry's **MLP Grad Steering** editor, which writes through a *different*
frozen object (the original 1×128 `MLPExtractor` on GRU `h`). Arm labels here carry "· MLP probe" explicitly.

In [ ]:
# [11] The same editors through the MLP probes. Emphasis: Latent Grad (pause–optimize–resume).
for p in PROBE["mlp"].parameters():
    p.requires_grad_(False)
for k_ in PAUSE_KS:
    for p in LAT_PROBES[k_]["mlp"].parameters():
        p.requires_grad_(False)
print("held-out MLP R² at the steering points:",
      " | ".join(f"(L3, τ={taus_full[k_]:.2f}): {LAT_PROBES[k_]['mlp_r2']:.3f}" for k_ in PAUSE_KS),
      f"| activations state: {PROBE['mlp_r2']:.3f}")

def input_grad_steer_mlp(lam, n=1, steps=STEER_STEPS, lr=STEER_LR):
    mlp = PROBE["mlp"]
    delta = torch.zeros(N, n, R, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    for it in range(steps):
        frames = torch.cat([hist[:, :ef - n], (hist[:, ef - n:] + delta).clamp(0, 1)], dim=1)
        read = mlp(activations(state_from_frames(frames)))
        loss = ((read - target4) ** 2).sum(-1).mean() + lam * (delta ** 2).sum((-1, -2)).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        frames = torch.cat([hist[:, :ef - n], (hist[:, ef - n:] + delta).clamp(0, 1)], dim=1)
        state = state_from_frames(frames)
        resid_after = (mlp(activations(state)) - target4).norm(dim=-1).mean().item()
        resid_before = (mlp(activations(state_unsteered)) - target4).norm(dim=-1).mean().item()
    return dict(state=state, delta=(frames[:, -1] - base_frame).cpu().numpy(),
                resid_before=resid_before, resid_after=resid_after)

def latent_grad_steer_mlp(k_pause, lam=0.0, steps=STEER_STEPS, lr=0.05, seed=456):
    """Identical to cell [8]'s latent_grad_steer, with the linear readout replaced by the (L3, τ_pause) MLP."""
    mlp = LAT_PROBES[k_pause]["mlp"]
    buffer = state_unsteered.obs_buffer
    cur, nxt = model._window_tokens(buffer)
    attn = model._window_attn_mask(full_len, DEVICE)
    gen = torch.Generator().manual_seed(seed)
    x = torch.randn(N, R, generator=gen).to(DEVICE)

    def v_at(xc, k):
        tau_t = torch.zeros(N, W, device=DEVICE); tau_t[:, -1] = taus_full[k]
        nxt_k = torch.cat([nxt[:, :-1], xc.unsqueeze(1)], dim=1)
        return model._denoise(cur, nxt_k, tau_t, attn)[:, -1]

    def probe_read(xc):
        tau_t = torch.zeros(N, W, device=DEVICE); tau_t[:, -1] = taus_full[k_pause]
        nxt_k = torch.cat([nxt[:, :-1], xc.unsqueeze(1)], dim=1)
        sink = []
        model._trunk(cur, nxt_k, tau_t, attn, resid_sink=sink)
        return mlp(sink[L_STEER][:, -1])

    with torch.no_grad():
        for k in range(k_pause):
            x = x + (taus_full[k + 1] - taus_full[k]) * v_at(x, k)
        resid_before = (probe_read(x) - target4).norm(dim=-1).mean().item()
    delta = torch.zeros_like(x, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    for it in range(steps):
        loss = ((probe_read(x + delta) - target4) ** 2).sum(-1).mean() + lam * (delta ** 2).sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        x = (x + delta).detach()
        resid_after = (probe_read(x) - target4).norm(dim=-1).mean().item()
        dn = float(delta.norm(dim=-1).mean())
        for k in range(k_pause, S):
            x = x + (taus_full[k + 1] - taus_full[k]) * v_at(x, k)
        frame = model._from_diff(x).clamp(0, 1)
    return frame, resid_before, resid_after, dn

name_ig_mlp = f"Input Grad · MLP probe (n=1, λ={LAM_MAIN})"
a = input_grad_steer_mlp(LAM_MAIN)
ARMS[name_ig_mlp] = a
ROLLS[name_ig_mlp] = rollout_from(a["state"])
CARDS[name_ig_mlp] = edit_scorecard(ROLLS[name_ig_mlp], ZONES, gt_roll)
c_o, a_o = cos_rows(a["delta"])
print(f"{name_ig_mlp:38s} probe residual {a['resid_before']:.3f} → {a['resid_after']:.3f} sim-units | "
      f"cos(δ, Δ_true) {c_o:+.3f} ({a_o:.0f}°) | Edit Index {CARDS[name_ig_mlp]['edit_index']:+.2f}")

MLP_LAT_FRAMES = {}
for k_ in PAUSE_KS:
    name = f"Latent Grad · MLP probe @(L3, τ={taus_full[k_]:.2f})"
    frame, r0, r1, dn = latent_grad_steer_mlp(k_)
    MLP_LAT_FRAMES[name] = frame.cpu().numpy()
    ROLLS[name] = rollout_from_frame(frame)
    CARDS[name] = edit_scorecard(ROLLS[name], ZONES, gt_roll)
    print(f"{name:38s} probe residual {r0:.3f} → {r1:.3f} sim-units | ‖δ‖ (diff space) {dn:.3f} | "
          f"Edit Index {CARDS[name]['edit_index']:+.2f} | collateral {CARDS[name]['collateral_rmse']:.3f}")

In [ ]:
# [12] Fig 8 — MLP-probe latent-steered frames; MLP-vs-linear scorecard; Fig 9 waterfall.
MLP_LAT_NAMES = [f"Latent Grad · MLP probe @(L3, τ={taus_full[k_]:.2f})" for k_ in PAUSE_KS]

fig, axes = plt.subplots(len(SAMPLES), 3, figsize=(4.6 * 3, 2.9 * len(SAMPLES)), sharex=True, sharey=True)
for r, smp in enumerate(SAMPLES):
    for c, name in enumerate(MLP_LAT_NAMES):
        ax = axes[r, c]
        ax.plot(x_axis, unguided_f0[smp], color="#999999", lw=1.0)
        ax.plot(x_axis, ZONES.gt_edited[smp], color="#009E73", lw=1.4, ls="--")
        ax.plot(x_axis, MLP_LAT_FRAMES[name][smp], color="#D55E00", lw=1.4)
        ax.fill_between(x_axis, 0, 1, where=ZONES.target[smp], color="#009E73", alpha=0.08)
        ax.fill_between(x_axis, 0, 1, where=ZONES.ghost[smp], color="#FF5252", alpha=0.08)
        ax.set_ylim(-0.05, 1.1)
        if r == 0: ax.set_title(name, fontsize=9.5)
        if c == 0: ax.set_ylabel(f"sample {smp}\n(teleport {teleport[smp]:.1f})\nintensity", fontsize=9)
        if r == len(SAMPLES) - 1: ax.set_xlabel("ray")
        style_ax(ax)
handles = [Line2D([0], [0], color="#999999", lw=1.5), Line2D([0], [0], color="#D55E00", lw=1.5),
           Line2D([0], [0], color="#009E73", ls="--", lw=1.5),
           Line2D([0], [0], color="#009E73", lw=6, alpha=0.2), Line2D([0], [0], color="#FF5252", lw=6, alpha=0.2)]
labels = ["unguided sample (control, gray)", "MLP-probe latent-steered resumed frame (orange)",
          "clean edited-world render", "target rays (shaded)", "ghost rays (shaded)"]
fig.legend(handles, labels, loc="upper center", ncol=5, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle(f"Fig 8 — pause–optimize–resume latent steering through the MLP probe, by pause time "
             f"({MODEL_LABEL}, probe at residual point 3)", y=1.06, fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{OUT}/fig8_mlp_latent_frames.png", dpi=150, bbox_inches="tight"); plt.show()

# MLP-vs-linear scorecard, matched rows
ORDER3 = ["Unsteered", "Unguided sample @step0 (control)",
          f"Input Grad (n=1, λ={LAM_MAIN})", name_ig_mlp]
for k_ in PAUSE_KS:
    ORDER3 += [f"Latent Grad @(L3, τ={taus_full[k_]:.2f})", f"Latent Grad · MLP probe @(L3, τ={taus_full[k_]:.2f})"]
ORDER3 += ["Render write @1 (oracle)"]
hdr = ("| arm | Edit Index (−1…+1) | edit-frame RMSE | target RMSE | ghost RMSE | collateral RMSE "
       "| GT-traj RMSE | fidelity ratio |\n|---|---|---|---|---|---|---|---|\n")
rows = []
for name in ORDER3:
    c = CARDS[name]
    rows.append(f"| {name} | {c['edit_index']:+.2f} | {c['edit_frame_rmse']:.3f} | {c['target_rmse']:.3f} "
                f"| {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} "
                f"| {fidelity_ratio(c, CARDS['Unsteered']):.2f} |")
display(Markdown("**§4 scorecard — MLP-probe arms interleaved with their linear counterparts** (same "
                 "conventions/caveats as the cell [6] table)\n\n" + hdr + "\n".join(rows)))

WF4 = ["Unsteered", name_ig_mlp] + MLP_LAT_NAMES \
      + [f"Latent Grad @(L3, τ={taus_full[4]:.2f})", "Render write @1 (oracle)"]
titles = ["GT (sim clean obs)"] + [f"{n}\nEdit Index {CARDS[n]['edit_index']:+.2f}" for n in WF4]
bodies = [gt_roll] + [ROLLS[n] for n in WF4]
waterfall_grid(titles, bodies, SAMPLES,
               f"Fig 9 — free-run waterfalls: MLP-probe steering arms vs the linear Latent Grad and references "
               f"({MODEL_LABEL})",
               f"{OUT}/fig9_mlp_waterfall.png")

In [ ]:
# [13] Two follow-ups (Sevan, 2026-08-11): (a) Input Grad · MLP probe over the WHOLE window (n=4);
#      (b) "Counterfactual window write (n=4, oracle)" — the intelligent whole-window overwrite: all 4 window
#      frames replaced by CLEAN renders of the counterfactual world in which the edited object travels with
#      ITS OWN velocity along a path offset by a constant Δ = tgt − (pre_pos + v·dt), so it arrives exactly at
#      the teleport target at frame ef; the other object keeps its true trajectory. Velocity-consistent
#      evidence on every frame — the DiT analogue of the registry's transformer "History overwrite (n frames)".
from pim.simulator.renderer import render_frame
from pim.simulator.sim import SimConfig

# (a) whole-window MLP input grad
name_ig_mlp4 = f"Input Grad · MLP probe (n=4 whole window, λ={LAM_MAIN})"
a4 = input_grad_steer_mlp(LAM_MAIN, n=W)
ARMS[name_ig_mlp4] = a4
ROLLS[name_ig_mlp4] = rollout_from(a4["state"])
CARDS[name_ig_mlp4] = edit_scorecard(ROLLS[name_ig_mlp4], ZONES, gt_roll)
c_o4, a_o4 = cos_rows(a4["delta"])
print(f"{name_ig_mlp4:48s} probe residual {a4['resid_before']:.3f} → {a4['resid_after']:.3f} sim-units | "
      f"cos(δ newest frame, Δ_true) {c_o4:+.3f} ({a_o4:.0f}°) | "
      f"Edit Index {CARDS[name_ig_mlp4]['edit_index']:+.2f}")

# (b) counterfactual window write
dt = float(sim["dt"])
cfg_r = SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                  n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=dt, obs_res=sim["obs_res"],
                  refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                  obs_noise_std=0.0, boundary="open", always_in_frustum=False,
                  soft_edge=sim.get("soft_edge", 0.0), soft_shading=sim.get("soft_shading", "flat"),
                  soft_psf_sigma=sim.get("soft_psf_sigma", 0.0),
                  soft_occlusion_temp=sim.get("soft_occlusion_temp", 0.0))
refl = np.linspace(sim["refl_min"], sim["refl_max"], N_OBJ).astype(np.float32)
rad = np.full(N_OBJ, sim["radius"], np.float32)
ar = np.arange(N)
offset = tgt_pos[ar, oe] - (pre_pos[ar, oe] + pre_vel[ar, oe] * dt)     # (N, 2), constant per sample
cf_frames = np.zeros((N, W, R), np.float32)
for i in range(N):
    for j, t in enumerate(range(ef - W, ef)):
        pos_t = edits.positions[i, t, :N_OBJ].astype(np.float32).copy()
        pos_t[oe[i]] += offset[i]
        _, _, inten = render_frame(pos_t, rad, refl, cfg_r)
        cf_frames[i, j] = inten
# construction check: the offset trajectory\'s next step lands exactly on the teleport target
land = pre_pos[ar, oe] + offset + pre_vel[ar, oe] * dt
print(f"construction check: max |landing − target| = {np.abs(land - tgt_pos[ar, oe]).max():.2e} sim-units")

name_cf = "Counterfactual window write (n=4, oracle)"
hist_cf = torch.from_numpy(cf_frames).float().to(DEVICE)
ROLLS[name_cf] = rollout_from(state_from_frames(hist_cf))
CARDS[name_cf] = edit_scorecard(ROLLS[name_cf], ZONES, gt_roll)
print(f"{name_cf:48s} Edit Index {CARDS[name_cf]['edit_index']:+.2f} | "
      f"collateral {CARDS[name_cf]['collateral_rmse']:.3f}")

ORDER4 = ["Unsteered", f"Input Grad · MLP probe (n=1, λ={LAM_MAIN})", name_ig_mlp4,
          "Render write @1 (oracle)", name_cf]
hdr = ("| arm | Edit Index (−1…+1) | edit-frame RMSE | target RMSE | ghost RMSE | collateral RMSE "
       "| GT-traj RMSE | fidelity ratio |\n|---|---|---|---|---|---|---|---|\n")
rows = []
for nm in ORDER4:
    c = CARDS[nm]
    rows.append(f"| {nm} | {c['edit_index']:+.2f} | {c['edit_frame_rmse']:.3f} | {c['target_rmse']:.3f} "
                f"| {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} "
                f"| {fidelity_ratio(c, CARDS['Unsteered']):.2f} |")
display(Markdown("**§4 scorecard — whole-window arms vs their single-frame counterparts**\n\n"
                 + hdr + "\n".join(rows)))

WF5 = ["Unsteered", f"Input Grad · MLP probe (n=1, λ={LAM_MAIN})", name_ig_mlp4,
       "Render write @1 (oracle)", name_cf]
titles = ["GT (sim clean obs)"] + [f"{nm}\nEdit Index {CARDS[nm]['edit_index']:+.2f}" for nm in WF5]
bodies = [gt_roll] + [ROLLS[nm] for nm in WF5]
waterfall_grid(titles, bodies, SAMPLES,
               f"Fig 10 — free-run waterfalls: whole-window MLP input steering and the velocity-consistent "
               f"counterfactual window write ({MODEL_LABEL})",
               f"{OUT}/fig10_window_write_waterfall.png")

## Current results (updated 2026-08-11; latent steering + full-Euler + MLP-probe variants same day)

- **Input Grad (history write, linear probe) replicates the transformer/GRU negative**: probe residual 3.37 → 0.05–0.14,
  cos(δ*, Δ_true) ≈ +0.11…+0.15, Edit Index −0.65 → −0.52, fidelity 1.00. **Whole-window (n=4): −0.52, = n=1.**
- **Per-step guided sampling fails at every strength; schedule (full vs early-τ) does not matter**: index
  plateaus at −0.05 with collateral 0.13 → 0.45–0.55 — degradation, not landing.
- **Pause–optimize–resume latent steering (linear probe)**: verified probes (L=3, linear R² 0.70), iterate
  driven to exact readout → Edit Index **−0.18 / −0.20 / −0.26** vs unguided control −0.51, collateral
  0.28–0.35. **Mechanism is duplication, not relocation** (Figs 5–6): new persistent target band + surviving
  ghost (the ghost lives in the clean context tokens, which iterate steering never touches).
- **Full-Euler robustness (cell [10], Fig 7): every verdict is rollout-mode-invariant** (Latent Grad
  −0.18/−0.20/−0.26 identical; oracle +0.13; sampling diffuses unedited belief slightly: unsteered −0.65 → −0.51).
- **MLP-probe variants (cells [11]–[12]; post-fix probes, held-out R² 0.85–0.90):**
  - On the **history-write surface the MLP gradient is markedly more semantic**: Input Grad · MLP reaches
    **−0.31** (vs linear −0.52 — the best history-write result on any architecture in the thread), cos(δ, Δ_true)
    +0.22, ghost RMSE 0.530 → 0.469, and the Fig 9 waterfall shows visible **ghost dimming plus a new
    target-side band** from a clean-frame edit. Notably the MLP readout is only driven to 0.40 sim-units
    (nonconvex) yet accomplishes more than the linear probe's exact 0.14 — what it does move is more meaningful.
  - On the **latent surface probe capacity is irrelevant**: MLP −0.29/−0.22/−0.21 vs linear −0.18/−0.20/−0.26 —
    the ≈ −0.2 plateau stands (slightly smaller ‖δ‖ and collateral, fidelity ≈ 1.01).
- **Same surface, oracle content: +0.12** (Render write @1) — between GRU (−0.01) and transformer W16 (+0.27),
  as the 4-frame buffer predicts.
- **Whole-window arms (cell [13], Fig 10):**
  - **Input Grad · MLP (n=4 whole window): −0.31, identical to n=1** (marginally better fidelity 0.97) — even
    the best probe gradient does not exploit the wider write surface.
  - **Counterfactual window write (n=4, oracle) — the edit essentially fully lands: Edit Index +0.71**, target
    RMSE 0.092, ghost RMSE 0.096, collateral 0.110 (= unsteered baseline), GT-traj RMSE **0.206 — better than
    the unsteered rollout's 0.303** (fidelity 0.68). All four window frames replaced by clean renders of the
    velocity-consistent counterfactual (edited object's own velocity, path offset by constant
    Δ = tgt − (pre_pos + v·dt); construction lands on target to 5e-7). The waterfall shows the edited object
    ON the target line with GT-matched motion and no ghost. **The Render write @1 ceiling of +0.12 was never
    belief inertia — it was conflicting velocity evidence**: one time-offset frame against three unedited ones;
    four consistent frames remove the conflict entirely.

## Summary (interpretation — clearly marked as such)

The ladder on one probe family: raw input gradients (linear −0.52 / MLP −0.31, partially semantic), per-step
guidance kicks (−0.05, destroyed), pause–optimize–resume (≈ −0.2, duplicated object) — invariant to how the
rollout is generated. Two structural readings: (1) probe *capacity* matters on the clean input surface (the
MLP's nonlinear feature structure constrains its gradient toward content) but **not** at the iterate — evidence
that the latent-steering plateau is set by **belief dynamics** (the untouched clean context tokens carrying the
ghost), not by readout quality; (2) the denoiser still adds a unique partial-landing channel no other
architecture has. Escalations if this line continues: SDEdit-style re-noise of the *history* (the one surface
now shown to respond to a good gradient), repeated latent edits over several generated frames, or a
render-space objective on the resumed frame.